In [1]:
# Cell 1 - Imports and paths
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_RAW  = Path('../data/raw')
DATA_PROC = Path('../data/processed')
SET_A     = DATA_RAW / 'training_setA'
SET_B     = DATA_RAW / 'training_setB'

# Make sure processed folder exists
DATA_PROC.mkdir(parents=True, exist_ok=True)

print("Libraries imported successfully")
print(f"Raw data path exists:       {DATA_RAW.exists()}")
print(f"Processed data path exists: {DATA_PROC.exists()}")

Libraries imported successfully
Raw data path exists:       True
Processed data path exists: True


In [2]:
# Reload the full dataset

def load_dataset(folder_path, dataset_name):
    all_patients = []
    files = list(folder_path.glob('*.psv'))
    
    for i, file in enumerate(files):
        df = pd.read_csv(file, sep='|')
        df['patient_id'] = file.stem
        df['dataset'] = dataset_name
        all_patients.append(df)
        
        if (i + 1) % 5000 == 0:
            print(f"  Loaded {i + 1}/{len(files)} files...")
    
    return pd.concat(all_patients, ignore_index=True)

print("Loading Set A...")
df_a = load_dataset(SET_A, 'A')
print(f"Set A: {df_a.shape[0]:,} rows")

print("\nLoading Set B...")
df_b = load_dataset(SET_B, 'B')
print(f"Set B: {df_b.shape[0]:,} rows")

df = pd.concat([df_a, df_b], ignore_index=True)
print(f"\nFull dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loading Set A...
  Loaded 5000/20336 files...
  Loaded 10000/20336 files...
  Loaded 15000/20336 files...
  Loaded 20000/20336 files...
Set A: 790,215 rows

Loading Set B...
  Loaded 5000/20000 files...
  Loaded 10000/20000 files...
  Loaded 15000/20000 files...
  Loaded 20000/20000 files...
Set B: 761,995 rows

Full dataset: 1,552,210 rows x 43 columns


In [3]:
# Feature set
# Every decision here is backed by our exploration findings

# Core vitals - recorded hourly, under 31% missing
VITAL_FEATURES = [
    'HR',       # Heart rate - elevated in sepsis
    'O2Sat',    # Oxygen saturation - drops in sepsis  
    'SBP',      # Systolic blood pressure - drops in sepsis
    'MAP',      # Mean arterial pressure - drops in sepsis
    'DBP',      # Diastolic blood pressure
    'Resp',     # Respiratory rate - STRONGEST pre-sepsis signal
    'Temp',     # Temperature - elevated in sepsis
]

# Key lab values - sparse but trend-informative
LAB_FEATURES = [
    'Lactate',      # Waste product - rises when cells lack oxygen
    'WBC',          # White blood cells - rises with infection
    'Creatinine',   # Kidney stress marker
    'Glucose',      # Blood sugar - stress hyperglycemia in sepsis
    'pH',           # Blood acidity - drops as sepsis progresses
    'Hgb',          # Haemoglobin - oxygen carrying capacity
]

# Demographics - always available, static throughout stay
DEMO_FEATURES = [
    'Age',          # Elderly patients higher risk
    'Gender',       # Biological differences in sepsis response
    'HospAdmTime',  # Time between hospital and ICU admission
    'ICULOS',       # How long they have been in ICU
]

# Missingness indicators - one per lab value
# Binary: 1 = lab was recorded this hour, 0 = not recorded
OBS_FEATURES = [f'{lab}_obs' for lab in LAB_FEATURES]

# All features combined
ALL_FEATURES = VITAL_FEATURES + LAB_FEATURES + DEMO_FEATURES + OBS_FEATURES

print("=" * 50)
print("FEATURE SET SUMMARY")
print("=" * 50)
print(f"Vital features:        {len(VITAL_FEATURES):>3} {VITAL_FEATURES}")
print(f"Lab features:          {len(LAB_FEATURES):>3} {LAB_FEATURES}")
print(f"Demographic features:  {len(DEMO_FEATURES):>3} {DEMO_FEATURES}")
print(f"Missingness indicators:{len(OBS_FEATURES):>3} {OBS_FEATURES}")
print(f"\nTotal features:        {len(ALL_FEATURES):>3}")
print(f"Target variable:       SepsisLabel")

FEATURE SET SUMMARY
Vital features:          7 ['HR', 'O2Sat', 'SBP', 'MAP', 'DBP', 'Resp', 'Temp']
Lab features:            6 ['Lactate', 'WBC', 'Creatinine', 'Glucose', 'pH', 'Hgb']
Demographic features:    4 ['Age', 'Gender', 'HospAdmTime', 'ICULOS']
Missingness indicators:  6 ['Lactate_obs', 'WBC_obs', 'Creatinine_obs', 'Glucose_obs', 'pH_obs', 'Hgb_obs']

Total features:         23
Target variable:       SepsisLabel


In [4]:
# Create prediction horizon label
# Our target: will sepsis occur within the NEXT 6 hours?
# This is the core of our "early warning" system

print("=" * 50)
print("CREATING 6-HOUR PREDICTION HORIZON LABEL")
print("=" * 50)

print("\nWhat we're doing:")
print("  Original SepsisLabel = 1 means sepsis IS happening NOW")
print("  Our new label = 1 means sepsis will happen in NEXT 6 hours")
print("  This gives clinicians time to intervene BEFORE onset\n")

def create_prediction_horizon(patient_df, horizon=6):
    """
    For each hour, look ahead 'horizon' hours.
    If sepsis occurs within that window, label this hour as 1.
    
    Example with horizon=6:
    Hour 20: SepsisLabel=0, but sepsis at hour 24
             → our label = 1 (sepsis in 4 hours, within 6h window)
    Hour 14: SepsisLabel=0, sepsis at hour 24  
             → our label = 0 (sepsis in 10 hours, outside 6h window)
    """
    patient_df = patient_df.sort_values('ICULOS').copy()
    
    # Rolling max over next 'horizon' rows
    # shift(-horizon) looks horizon steps ahead
    patient_df['sepsis_in_6h'] = (
        patient_df['SepsisLabel']
        .rolling(window=horizon, min_periods=1)
        .max()
        .shift(-horizon)
        .fillna(0)
        .astype(int)
    )
    return patient_df

# Apply to each patient separately
print("Applying 6-hour horizon to all patients...")
df = df.groupby('patient_id', group_keys=False).apply(
    create_prediction_horizon
)

# Check the result
original_positives = df['SepsisLabel'].sum()
new_positives = df['sepsis_in_6h'].sum()

print(f"\nOriginal SepsisLabel positives:  {original_positives:>8,} ({original_positives/len(df)*100:.2f}%)")
print(f"New sepsis_in_6h positives:      {new_positives:>8,} ({new_positives/len(df)*100:.2f}%)")
print(f"\nIncrease: {new_positives - original_positives:,} more positive hours")
print("(Expected - each sepsis onset now also flags the 6 preceding hours)")

# Verify with one patient example
example_pid = df[df['SepsisLabel'] == 1]['patient_id'].iloc[0]
example = df[df['patient_id'] == example_pid][
    ['ICULOS', 'SepsisLabel', 'sepsis_in_6h']
].tail(15)
print(f"\nExample patient {example_pid} - last 15 hours:")
print(example.to_string(index=False))

CREATING 6-HOUR PREDICTION HORIZON LABEL

What we're doing:
  Original SepsisLabel = 1 means sepsis IS happening NOW
  Our new label = 1 means sepsis will happen in NEXT 6 hours
  This gives clinicians time to intervene BEFORE onset

Applying 6-hour horizon to all patients...

Original SepsisLabel positives:    27,916 (1.80%)
New sepsis_in_6h positives:        24,403 (1.57%)

Increase: -3,513 more positive hours
(Expected - each sepsis onset now also flags the 6 preceding hours)

Example patient p000009 - last 15 hours:
 ICULOS  SepsisLabel  sepsis_in_6h
    244            0             1
    245            0             1
    246            0             1
    247            0             1
    248            0             1
    249            1             1
    250            1             1
    251            1             1
    252            1             1
    253            1             0
    254            1             0
    255            1             0
    256            

In [5]:
# Fixed prediction horizon label

print("=" * 50)
print("FIXED 6-HOUR PREDICTION HORIZON LABEL")
print("=" * 50)

def create_prediction_horizon_fixed(patient_df, horizon=6):
    """
    Corrected version:
    For each hour, check if ANY of the next 'horizon' hours
    have SepsisLabel = 1.
    
    Uses a cleaner approach:
    - For each row, look at the NEXT horizon rows
    - If any of them have SepsisLabel=1, flag this row as 1
    - Also keep rows where SepsisLabel is already 1 as positive
    """
    patient_df = patient_df.sort_values('ICULOS').copy()
    labels = patient_df['SepsisLabel'].values
    n = len(labels)
    new_labels = np.zeros(n, dtype=int)
    
    for i in range(n):
        # Look ahead up to horizon steps
        lookahead_end = min(i + horizon + 1, n)
        # If any hour in the next 6 hours is sepsis → flag this hour
        if labels[i:lookahead_end].max() == 1:
            new_labels[i] = 1
    
    patient_df['sepsis_in_6h'] = new_labels
    return patient_df

# Apply to all patients
print("Applying fixed 6-hour horizon to all patients...")
print("(This may take 3-4 minutes for 40,336 patients)\n")

df = df.groupby('patient_id', group_keys=False).apply(
    create_prediction_horizon_fixed
)

# Check results
original_positives = df['SepsisLabel'].sum()
new_positives = df['sepsis_in_6h'].sum()

print(f"Original SepsisLabel positives:  {original_positives:>8,} ({original_positives/len(df)*100:.2f}%)")
print(f"New sepsis_in_6h positives:      {new_positives:>8,} ({new_positives/len(df)*100:.2f}%)")
print(f"Increase: +{new_positives - original_positives:,} more positive hours")
print("(Each sepsis onset now flags up to 6 preceding hours too)")

# Verify with same patient
example_pid = 'p000009'
example = df[df['patient_id'] == example_pid][
    ['ICULOS', 'SepsisLabel', 'sepsis_in_6h']
].tail(20)
print(f"\nExample patient {example_pid} - last 20 hours:")
print(example.to_string(index=False))

FIXED 6-HOUR PREDICTION HORIZON LABEL
Applying fixed 6-hour horizon to all patients...
(This may take 3-4 minutes for 40,336 patients)

Original SepsisLabel positives:    27,916 (1.80%)
New sepsis_in_6h positives:        41,995 (2.71%)
Increase: +14,079 more positive hours
(Each sepsis onset now flags up to 6 preceding hours too)

Example patient p000009 - last 20 hours:
 ICULOS  SepsisLabel  sepsis_in_6h
    239            0             0
    240            0             0
    241            0             0
    242            0             0
    243            0             1
    244            0             1
    245            0             1
    246            0             1
    247            0             1
    248            0             1
    249            1             1
    250            1             1
    251            1             1
    252            1             1
    253            1             1
    254            1             1
    255            1           

In [6]:
# Create missingness indicator features
# Binary columns: 1 = lab was recorded this hour, 0 = missing

print("=" * 50)
print("CREATING MISSINGNESS INDICATORS")
print("=" * 50)

print("\nWhat we're doing:")
print("  For each lab feature, create a new binary column")
print("  1 = the lab was actually measured this hour")
print("  0 = no measurement taken")
print("  This captures the clinical signal of 'a lab was ordered'\n")

for lab in LAB_FEATURES:
    obs_col = f'{lab}_obs'
    df[obs_col] = df[lab].notna().astype(int)
    n_recorded = df[obs_col].sum()
    pct = n_recorded / len(df) * 100
    print(f"  {obs_col:<20} {n_recorded:>8,} hours recorded ({pct:.1f}%)")

print(f"\nMissingness indicators created: {len(OBS_FEATURES)}")
print("Columns added:", OBS_FEATURES)

CREATING MISSINGNESS INDICATORS

What we're doing:
  For each lab feature, create a new binary column
  1 = the lab was actually measured this hour
  0 = no measurement taken
  This captures the clinical signal of 'a lab was ordered'

  Lactate_obs            41,446 hours recorded (2.7%)
  WBC_obs                99,447 hours recorded (6.4%)
  Creatinine_obs         94,616 hours recorded (6.1%)
  Glucose_obs           265,516 hours recorded (17.1%)
  pH_obs                107,573 hours recorded (6.9%)
  Hgb_obs               114,591 hours recorded (7.4%)

Missingness indicators created: 6
Columns added: ['Lactate_obs', 'WBC_obs', 'Creatinine_obs', 'Glucose_obs', 'pH_obs', 'Hgb_obs']


In [7]:
# Imputation
# Strategy:
# Step 1: Forward-fill within each patient
#         (carry last known value forward hour by hour)
# Step 2: Backward-fill within each patient  
#         (for missing values at START of stay)
# Step 3: Median imputation for anything still missing
#         (patients who never had a reading at all)

print("=" * 50)
print("IMPUTATION")
print("=" * 50)

print("\nStrategy:")
print("  Step 1: Forward-fill  — carry last known value forward")
print("  Step 2: Backward-fill — fill gaps at start of stay")
print("  Step 3: Median fill   — for features never recorded\n")

# Features to impute (vitals + labs only, not demographics or indicators)
IMPUTE_FEATURES = VITAL_FEATURES + LAB_FEATURES

# Check missing BEFORE imputation
print("Missing values BEFORE imputation:")
print(f"  {'Feature':<15} {'Missing':>10} {'%':>8}")
print("  " + "-" * 36)
for feat in IMPUTE_FEATURES:
    n_missing = df[feat].isna().sum()
    pct = n_missing / len(df) * 100
    print(f"  {feat:<15} {n_missing:>10,} {pct:>7.1f}%")

# Calculate medians from training data BEFORE filling
# Important: calculate on Set A only (our training set)
# so we don't leak Set B info into imputation
print("\nCalculating medians from Set A (training set only)...")
set_a_data = df[df['dataset'] == 'A']
MEDIANS = {}
for feat in IMPUTE_FEATURES:
    MEDIANS[feat] = set_a_data[feat].median()
    
print("Medians calculated:")
for feat, median in MEDIANS.items():
    print(f"  {feat:<15}: {median:.3f}")

IMPUTATION

Strategy:
  Step 1: Forward-fill  — carry last known value forward
  Step 2: Backward-fill — fill gaps at start of stay
  Step 3: Median fill   — for features never recorded

Missing values BEFORE imputation:
  Feature            Missing        %
  ------------------------------------
  HR                 153,399     9.9%
  O2Sat              202,736    13.1%
  SBP                226,265    14.6%
  MAP                193,270    12.5%
  DBP                486,554    31.3%
  Resp               238,335    15.4%
  Temp             1,026,984    66.2%
  Lactate          1,510,764    97.3%
  WBC              1,452,763    93.6%
  Creatinine       1,457,594    93.9%
  Glucose          1,286,694    82.9%
  pH               1,444,637    93.1%
  Hgb              1,437,619    92.6%

Calculating medians from Set A (training set only)...
Medians calculated:
  HR             : 84.000
  O2Sat          : 98.000
  SBP            : 118.500
  MAP            : 77.000
  DBP            : 58.500
  

In [8]:
# Apply imputation

print("Applying imputation to all patients...")
print("(This will take 3-4 minutes)\n")

def impute_patient(patient_df):
    """
    Apply 3-step imputation to a single patient's data.
    Must be done per-patient so forward-fill doesn't
    bleed values from one patient into another.
    """
    patient_df = patient_df.sort_values('ICULOS').copy()
    
    # Step 1: Forward-fill
    # If HR was 85 at hour 3 and missing at hour 4,
    # hour 4 becomes 85
    patient_df[IMPUTE_FEATURES] = (
        patient_df[IMPUTE_FEATURES].ffill()
    )
    
    # Step 2: Backward-fill
    # If HR is missing at hour 1 but recorded at hour 2,
    # hour 1 gets hour 2's value
    patient_df[IMPUTE_FEATURES] = (
        patient_df[IMPUTE_FEATURES].bfill()
    )
    
    return patient_df

# Apply per patient
df = df.groupby('patient_id', group_keys=False).apply(impute_patient)

# Step 3: Median fill for anything still missing
# (patients who NEVER had a reading for a feature)
print("Applying median fill for remaining missing values...")
for feat in IMPUTE_FEATURES:
    n_still_missing = df[feat].isna().sum()
    if n_still_missing > 0:
        df[feat] = df[feat].fillna(MEDIANS[feat])
        print(f"  {feat:<15}: filled {n_still_missing:,} remaining NaN with median {MEDIANS[feat]:.3f}")

# Verify - check missing AFTER imputation
print("\nMissing values AFTER imputation:")
print(f"  {'Feature':<15} {'Missing':>10} {'%':>8}")
print("  " + "-" * 36)
all_clean = True
for feat in IMPUTE_FEATURES:
    n_missing = df[feat].isna().sum()
    pct = n_missing / len(df) * 100
    status = "✓" if n_missing == 0 else "✗ PROBLEM"
    print(f"  {feat:<15} {n_missing:>10,} {pct:>7.1f}%  {status}")
    if n_missing > 0:
        all_clean = False

print(f"\n{'All features clean - ready for scaling!' if all_clean else 'WARNING: Some features still have missing values!'}")

Applying imputation to all patients...
(This will take 3-4 minutes)

Applying median fill for remaining missing values...
  HR             : filled 142 remaining NaN with median 84.000
  O2Sat          : filled 374 remaining NaN with median 98.000
  SBP            : filled 10,976 remaining NaN with median 118.500
  MAP            : filled 2,698 remaining NaN with median 77.000
  DBP            : filled 259,250 remaining NaN with median 58.500
  Resp           : filled 1,999 remaining NaN with median 18.000
  Temp           : filled 7,430 remaining NaN with median 37.060
  Lactate        : filled 986,175 remaining NaN with median 1.800
  WBC            : filled 65,619 remaining NaN with median 10.800
  Creatinine     : filled 50,842 remaining NaN with median 0.900
  Glucose        : filled 42,046 remaining NaN with median 124.000
  pH             : filled 736,170 remaining NaN with median 7.390
  Hgb            : filled 61,519 remaining NaN with median 10.400

Missing values AFTER imput

In [9]:
# Feature scaling
# Scale all features to a similar range for LSTM stability
# LSTMs are sensitive to feature scale - a feature ranging 0-300
# will dominate one ranging 0-1 without scaling

print("=" * 50)
print("FEATURE SCALING")
print("=" * 50)

print("""
Why we scale:
  HR ranges 20-280 bpm
  pH ranges 6.8-7.8
  Without scaling, HR dominates the LSTM weights
  With scaling, all features contribute equally

Method: MinMaxScaler
  Transforms each feature to range [0, 1]
  Formula: (value - min) / (max - min)
  
Important: fit scaler on Set A ONLY
  Then transform both Set A and Set B
  Prevents data leakage from test set into scaling
""")

from sklearn.preprocessing import MinMaxScaler

# Features to scale (not demographics like Gender, 
# and not binary indicators - they're already 0/1)
SCALE_FEATURES = VITAL_FEATURES + LAB_FEATURES + ['Age', 'HospAdmTime', 'ICULOS']

# Fit scaler on Set A only
print("Fitting scaler on Set A (training data only)...")
set_a_df = df[df['dataset'] == 'A']
scaler = MinMaxScaler()
scaler.fit(set_a_df[SCALE_FEATURES])

print("Scaler fitted. Feature ranges from Set A:")
print(f"\n  {'Feature':<15} {'Min':>10} {'Max':>10} {'Range':>10}")
print("  " + "-" * 48)
for i, feat in enumerate(SCALE_FEATURES):
    fmin = scaler.data_min_[i]
    fmax = scaler.data_max_[i]
    print(f"  {feat:<15} {fmin:>10.3f} {fmax:>10.3f} {fmax-fmin:>10.3f}")

# Transform both sets
print("\nTransforming all data...")
df[SCALE_FEATURES] = scaler.transform(df[SCALE_FEATURES])

# Verify scaling worked
print("\nAfter scaling - sample statistics:")
print(f"  {'Feature':<15} {'Min':>8} {'Max':>8} {'Mean':>8}")
print("  " + "-" * 42)
for feat in SCALE_FEATURES[:5]:  # Show first 5
    print(f"  {feat:<15} "
          f"{df[feat].min():>8.3f} "
          f"{df[feat].max():>8.3f} "
          f"{df[feat].mean():>8.3f}")
print("  ... (all features scaled to [0,1])")

# Save scaler for later use in API
import pickle
scaler_path = Path('../models/scaler.pkl')
scaler_path.parent.mkdir(exist_ok=True)
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"\nScaler saved to {scaler_path}")

FEATURE SCALING

Why we scale:
  HR ranges 20-280 bpm
  pH ranges 6.8-7.8
  Without scaling, HR dominates the LSTM weights
  With scaling, all features contribute equally

Method: MinMaxScaler
  Transforms each feature to range [0, 1]
  Formula: (value - min) / (max - min)

Important: fit scaler on Set A ONLY
  Then transform both Set A and Set B
  Prevents data leakage from test set into scaling

Fitting scaler on Set A (training data only)...
Scaler fitted. Feature ranges from Set A:

  Feature                Min        Max      Range
  ------------------------------------------------
  HR                  20.000    280.000    260.000
  O2Sat               20.000    100.000     80.000
  SBP                 22.000    281.000    259.000
  MAP                 20.000    300.000    280.000
  DBP                 20.000    298.000    278.000
  Resp                 1.000     69.000     68.000
  Temp                20.900     42.220     21.320
  Lactate              0.200     31.000     30.80

In [10]:
# Create sliding window sequences

print("=" * 50)
print("CREATING SLIDING WINDOW SEQUENCES")
print("=" * 50)

WINDOW_SIZE = 6

def create_sequences(patient_df, window_size, feature_cols):
    patient_df = patient_df.sort_values('ICULOS').copy()
    values = patient_df[feature_cols].values
    labels = patient_df['sepsis_in_6h'].values
    n = len(values)
    
    if n < window_size:
        return None, None
    
    X, y = [], []
    for i in range(window_size, n + 1):
        window = values[i - window_size:i]
        label = labels[i - 1]
        X.append(window)
        y.append(label)
    
    return np.array(X), np.array(y)

# Test on one patient first
test_pid = 'p000001'
test_df = df[df['patient_id'] == test_pid]
X_test, y_test = create_sequences(test_df, WINDOW_SIZE, ALL_FEATURES)

print(f"Test on patient {test_pid}:")
print(f"  Patient hours: {len(test_df)}")
print(f"  Sequences generated: {X_test.shape[0]}")
print(f"  Sequence shape: {X_test.shape[1:]} "
      f"(window_size x features)")
print(f"  Labels shape: {y_test.shape}")
print(f"  Sample labels: {y_test[:10]}")

CREATING SLIDING WINDOW SEQUENCES
Test on patient p000001:
  Patient hours: 54
  Sequences generated: 49
  Sequence shape: (6, 23) (window_size x features)
  Labels shape: (49,)
  Sample labels: [0 0 0 0 0 0 0 0 0 0]


In [11]:
# Apply sequence creation to all patients
# Split by dataset: Set A = train, Set B = test

print("=" * 50)
print("BUILDING FULL SEQUENCE DATASET")
print("=" * 50)

def build_sequences_for_group(group_df, window_size, feature_cols):
    all_X, all_y = [], []
    patients = group_df['patient_id'].unique()
    
    for i, pid in enumerate(patients):
        patient_df = group_df[group_df['patient_id'] == pid]
        X, y = create_sequences(patient_df, window_size, feature_cols)
        
        if X is not None:
            all_X.append(X)
            all_y.append(y)
        
        if (i + 1) % 5000 == 0:
            print(f"  Processed {i + 1}/{len(patients)} patients...")
    
    return np.concatenate(all_X), np.concatenate(all_y)

# Set A → training data
print("Building training sequences (Set A)...")
df_a = df[df['dataset'] == 'A']
X_train, y_train = build_sequences_for_group(
    df_a, WINDOW_SIZE, ALL_FEATURES
)
print(f"Training set: {X_train.shape[0]:,} sequences")

# Set B → test data  
print("\nBuilding test sequences (Set B)...")
df_b = df[df['dataset'] == 'B']
X_test, y_test = build_sequences_for_group(
    df_b, WINDOW_SIZE, ALL_FEATURES
)
print(f"Test set: {X_test.shape[0]:,} sequences")

print(f"\nShapes:")
print(f"  X_train: {X_train.shape} "
      f"(sequences x timesteps x features)")
print(f"  y_train: {y_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_test:  {y_test.shape}")

print(f"\nClass distribution:")
print(f"  Train — Sepsis: {y_train.sum():,} "
      f"({y_train.mean()*100:.2f}%)")
print(f"  Test  — Sepsis: {y_test.sum():,} "
      f"({y_test.mean()*100:.2f}%)")

BUILDING FULL SEQUENCE DATASET
Building training sequences (Set A)...
  Processed 5000/20336 patients...
  Processed 10000/20336 patients...
  Processed 15000/20336 patients...
  Processed 20000/20336 patients...
Training set: 688,535 sequences

Building test sequences (Set B)...
  Processed 5000/20000 patients...
  Processed 10000/20000 patients...
  Processed 15000/20000 patients...
  Processed 20000/20000 patients...
Test set: 661,995 sequences

Shapes:
  X_train: (688535, 6, 23) (sequences x timesteps x features)
  y_train: (688535,)
  X_test:  (661995, 6, 23)
  y_test:  (661995,)

Class distribution:
  Train — Sepsis: 23,496 (3.41%)
  Test  — Sepsis: 14,145 (2.14%)


In [14]:
# Cell - Find and fix remaining NaN values in sequences

print("Checking for NaN in training sequences...")
nan_count = np.isnan(X_train).sum()
print(f"Total NaN values in X_train: {nan_count:,}")

# Find which features have NaN
nan_per_feature = np.isnan(X_train).sum(axis=(0,1))
print("\nNaN per feature:")
for i, feat in enumerate(ALL_FEATURES):
    if nan_per_feature[i] > 0:
        print(f"  {feat}: {nan_per_feature[i]:,} NaN values")

# Fix: replace any remaining NaN with 0
# 0 = minimum of scaled range, safe default
print("\nReplacing NaN with 0...")
X_train = np.nan_to_num(X_train, nan=0.0)
X_test = np.nan_to_num(X_test, nan=0.0)

# Verify
nan_after = np.isnan(X_train).sum()
print(f"NaN remaining after fix: {nan_after} "
      f"{'✓' if nan_after == 0 else '✗'}")

Checking for NaN in training sequences...
Total NaN values in X_train: 18

NaN per feature:
  HospAdmTime: 18 NaN values

Replacing NaN with 0...
NaN remaining after fix: 0 ✓


In [15]:
# Apply SMOTE to training data only
# Never apply SMOTE to test data - test must reflect reality

print("=" * 50)
print("APPLYING SMOTE TO TRAINING DATA")
print("=" * 50)

print("""
Why SMOTE on training only:
  Test set must reflect real-world class distribution
  Applying SMOTE to test would give fake performance metrics
  
How SMOTE works:
  Creates synthetic minority class samples
  by interpolating between existing sepsis sequences
  Target: balance classes to improve model learning
""")

from imblearn.over_sampling import SMOTE

# SMOTE works on 2D data - we need to reshape
# (sequences, timesteps, features) → (sequences, timesteps*features)
n_train, timesteps, n_features = X_train.shape
X_train_2d = X_train.reshape(n_train, timesteps * n_features)

print(f"Before SMOTE:")
print(f"  Total sequences:  {n_train:,}")
print(f"  Sepsis (1):       {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"  No sepsis (0):    {(y_train==0).sum():,} ({(y_train==0).mean()*100:.2f}%)")

# Apply SMOTE
# sampling_strategy=0.2 means minority becomes 20% of majority
# Full 50/50 balance is too artificial for clinical data
print(f"\nApplying SMOTE (target: 20% sepsis ratio)...")
print("This will take a few minutes...")

smote = SMOTE(sampling_strategy=0.2, random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train_2d, y_train
)

# Reshape back to 3D
X_train_balanced = X_train_balanced.reshape(
    -1, timesteps, n_features
)

print(f"\nAfter SMOTE:")
print(f"  Total sequences:  {len(X_train_balanced):,}")
print(f"  Sepsis (1):       {y_train_balanced.sum():,} "
      f"({y_train_balanced.mean()*100:.2f}%)")
print(f"  No sepsis (0):    {(y_train_balanced==0).sum():,} "
      f"({(y_train_balanced==0).mean()*100:.2f}%)")
print(f"\nNew sepsis ratio: "
      f"{y_train_balanced.mean()*100:.1f}% "
      f"(was {y_train.mean()*100:.1f}%)")

APPLYING SMOTE TO TRAINING DATA

Why SMOTE on training only:
  Test set must reflect real-world class distribution
  Applying SMOTE to test would give fake performance metrics

How SMOTE works:
  Creates synthetic minority class samples
  by interpolating between existing sepsis sequences
  Target: balance classes to improve model learning

Before SMOTE:
  Total sequences:  688,535
  Sepsis (1):       23,496 (3.41%)
  No sepsis (0):    665,039 (96.59%)

Applying SMOTE (target: 20% sepsis ratio)...
This will take a few minutes...

After SMOTE:
  Total sequences:  798,046
  Sepsis (1):       133,007 (16.67%)
  No sepsis (0):    665,039 (83.33%)

New sepsis ratio: 16.7% (was 3.4%)


In [16]:
# Save everything to disk

print("=" * 50)
print("SAVING PROCESSED DATA")
print("=" * 50)

import pickle

# Save numpy arrays
print("Saving sequences...")
np.save(DATA_PROC / 'X_train.npy', X_train_balanced)
np.save(DATA_PROC / 'y_train.npy', y_train_balanced)
np.save(DATA_PROC / 'X_test.npy',  X_test)
np.save(DATA_PROC / 'y_test.npy',  y_test)

# Save unbalanced train too (useful for evaluation)
np.save(DATA_PROC / 'X_train_raw.npy', X_train)
np.save(DATA_PROC / 'y_train_raw.npy', y_train)

# Save feature list and metadata
metadata = {
    'feature_cols':    ALL_FEATURES,
    'vital_features':  VITAL_FEATURES,
    'lab_features':    LAB_FEATURES,
    'demo_features':   DEMO_FEATURES,
    'obs_features':    OBS_FEATURES,
    'window_size':     WINDOW_SIZE,
    'n_features':      len(ALL_FEATURES),
    'train_shape':     X_train_balanced.shape,
    'test_shape':      X_test.shape,
    'smote_ratio':     0.2,
    'scale_features':  SCALE_FEATURES,
}

with open(DATA_PROC / 'metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print("Files saved:")
for f in sorted(DATA_PROC.iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:<25} {size_mb:.1f} MB")

print(f"\nFinal summary:")
print(f"  Training sequences: {X_train_balanced.shape[0]:,}")
print(f"  Test sequences:     {X_test.shape[0]:,}")
print(f"  Sequence shape:     {X_train_balanced.shape[1:]} "
      f"(timesteps x features)")
print(f"  Training sepsis:    {y_train_balanced.mean()*100:.1f}%")
print(f"  Test sepsis:        {y_test.mean()*100:.1f}%")
print(f"\nPreprocessing complete!")

SAVING PROCESSED DATA
Saving sequences...
Files saved:
  metadata.pkl              0.0 MB
  X_test.npy                697.0 MB
  X_train.npy               840.2 MB
  X_train_raw.npy           724.9 MB
  y_test.npy                5.1 MB
  y_train.npy               6.1 MB
  y_train_raw.npy           5.3 MB

Final summary:
  Training sequences: 798,046
  Test sequences:     661,995
  Sequence shape:     (6, 23) (timesteps x features)
  Training sepsis:    16.7%
  Test sepsis:        2.1%

Preprocessing complete!


In [17]:
# Retrain SMOTE with lower ratio (10% instead of 20%)

print("Applying improved SMOTE (10% ratio)...")

from imblearn.over_sampling import SMOTE
import numpy as np

# Load raw unbalanced training data
X_train_raw = np.load(DATA_PROC / 'X_train_raw.npy')
y_train_raw = np.load(DATA_PROC / 'y_train_raw.npy')

# Fix NaN first
X_train_raw = np.nan_to_num(X_train_raw, nan=0.0)

n_train, timesteps, n_features = X_train_raw.shape
X_train_2d = X_train_raw.reshape(n_train, timesteps * n_features)

print(f"Before SMOTE:")
print(f"  Sepsis: {y_train_raw.sum():,} ({y_train_raw.mean()*100:.2f}%)")

# Lower ratio - 10% instead of 20%
smote = SMOTE(sampling_strategy=0.1, random_state=42)
X_balanced, y_balanced = smote.fit_resample(X_train_2d, y_train_raw)

X_balanced = X_balanced.reshape(-1, timesteps, n_features)

print(f"\nAfter SMOTE:")
print(f"  Total: {len(X_balanced):,}")
print(f"  Sepsis: {y_balanced.sum():,} ({y_balanced.mean()*100:.2f}%)")

# Save new balanced data
np.save(DATA_PROC / 'X_train_v2.npy', X_balanced)
np.save(DATA_PROC / 'y_train_v2.npy', y_balanced)
print("\nSaved as X_train_v2.npy and y_train_v2.npy")

Applying improved SMOTE (10% ratio)...
Before SMOTE:
  Sepsis: 23,496 (3.41%)

After SMOTE:
  Total: 731,542
  Sepsis: 66,503 (9.09%)

Saved as X_train_v2.npy and y_train_v2.npy
